# Wilderness Urban Interface

1. **Imports and Setup**  
   - Imports standard libraries (e.g., `os`, `sys`, `subprocess`, `logging`) and specialized libraries for geospatial data processing (e.g., `geopandas`, `rasterio`, `shapely`, `tqdm`, `pandas`, `matplotlib`).
   - Sets up XML parsing and parallel processing (using `ProcessPoolExecutor`).

2. **Configuration Variables**  
   - Defines key parameters such as the number of parallel workers, directory names for outputs (masked rasters, VRT, plot), and logging configurations.
   - Specifies the new VRT name and the plot output details.

3. **Logger Initialization**  
   - Configures logging to capture errors both to a log file (`masking_errors.log`) and to the console.
   - Sets a formatter to standardize log messages.

4. **VRT File Validation and Parsing**  
   - Validates that the specified VRT file exists.
   - Parses the VRT file using XML to extract a list of referenced raster file paths, handling relative paths appropriately.

5. **Loading the Boundary Shapefile**  
   - Validates the existence of the study area boundary shapefile and its required auxiliary files.
   - Loads the shapefile using GeoPandas to represent the study area boundary.

6. **Reprojecting the Study Area**  
   - Determines the CRS (Coordinate Reference System) from the first valid raster.
   - Reprojects the study area boundary to match the raster’s CRS to ensure spatial compatibility.

7. **Raster Classification**  
   - Iterates through each raster file and creates its bounding box.
   - Classifies each raster into either "fully inside" the study area (bounding box completely within the boundary) or "edge" (intersecting the boundary but not completely inside).

8. **Masking Edge Rasters in Parallel**  
   - For rasters that only partially intersect the study area ("edge rasters"), applies a mask to set cells outside the boundary to a designated nodata value.
   - Uses parallel processing to speed up the masking operation.
   - Writes the masked rasters to a dedicated output directory.

9. **Preparing Rasters for VRT Creation**  
   - Combines the list of "fully inside" rasters with the newly masked "edge" rasters.
   - Verifies that there is at least one raster to include in the new VRT.

10. **Creating a New VRT File**  
    - Writes the paths of the selected rasters to a temporary file.
    - Uses the GDAL utility `gdalbuildvrt` (via `subprocess.run`) to create a new VRT file that references only the selected rasters.
    - Cleans up the temporary file after successful VRT creation.

11. **Plotting Raster Bounding Boxes**  
    - Creates GeoDataFrames for both "fully inside" and "edge" rasters based on their bounding boxes.
    - Plots the study area boundary along with the raster bounding boxes using Matplotlib.
    - Saves the generated plot to a specified file location.

12. **Completion Notification**  
    - Outputs messages to the console indicating successful completion.
    - Informs the user where any error logs have been saved.

In [ ]:
import os
WRI_PROJECT_ROOT = os.environ.get("WRI_PROJECT_ROOT", "/home/shares/wwri-wildfire")

import os
import sys
import geopandas as gpd
import rasterio
from shapely.geometry import box
import traceback
from tqdm import tqdm
import subprocess
import rasterio.mask
from concurrent.futures import ProcessPoolExecutor, as_completed
from xml.etree import ElementTree as ET
import logging
import matplotlib.pyplot as plt  # Added for plotting
import pandas as pd  # **Ensure this line is present**


# ===========================
# CONFIGURATION VARIABLES
# ===========================

# Number of parallel workers for masking rasters
NUM_WORKERS = 50  # Adjust based on your CPU cores and workload

# Output directory for masked rasters
MASKED_DIR_NAME = "masked_rasters"

# Logging configuration
LOG_FILE = "masking_errors.log"

# New VRT output configuration
NEW_VRT_DIR = os.path.join(WRI_PROJECT_ROOT, 'data', 'infrastructure', 'intermediate')
NEW_VRT_NAME = "study_area_wui_map.vrt"

# Plot configuration
PLOT_OUTPUT_DIR = os.path.join(WRI_PROJECT_ROOT, 'data', 'infrastructure', 'intermediate')
PLOT_OUTPUT_NAME = "raster_bounding_boxes.png"

# ===========================
# SET UP LOGGER WITH FILE AND CONSOLE HANDLERS
# ===========================

logger = logging.getLogger()
logger.setLevel(logging.ERROR)  # Set the logging level to ERROR

# Create formatter
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

# File handler for logging errors to a file
file_handler = logging.FileHandler(LOG_FILE, mode='w')
file_handler.setLevel(logging.ERROR)
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# Console handler for printing errors to the console
console_handler = logging.StreamHandler(sys.stderr)
console_handler.setLevel(logging.ERROR)
console_handler.setFormatter(formatter)
logger.addHandler(console_handler)

# ===========================
# DEFINE PATHS AND VALIDATE VRT FILE
# ===========================

base_dir = WRI_PROJECT_ROOT
vrt_path = os.path.join(
    base_dir, "data/multi-domain-data/raw/wildland_urban_interface/raw/NA/mosaic/WUI.vrt"
)

if not os.path.exists(vrt_path):
    print(f"VRT file does not exist at the path: {vrt_path}")
    sys.exit(1)

# ===========================
# PARSE VRT FILE TO LIST REFERENCED RASTER FILES
# ===========================

def parse_vrt(vrt_file):
    try:
        tree = ET.parse(vrt_file)
        root = tree.getroot()
        raster_files = []
        for elem in root.iter():
            if elem.tag.endswith("SourceFilename"):
                raster_file = elem.text
                # Handle relative paths
                if not os.path.isabs(raster_file):
                    raster_file = os.path.join(os.path.dirname(vrt_file), raster_file)
                raster_files.append(os.path.normpath(raster_file))
        return raster_files
    except Exception as e:
        print(f"Error parsing VRT file: {e}")
        logger.error(f"Error parsing VRT file: {e}")
        traceback_str = traceback.format_exc()
        logger.error(traceback_str)
        sys.exit(1)

print("Parsing VRT file to list referenced raster files...")
raster_files = parse_vrt(vrt_path)
print(f"Total rasters referenced in VRT: {len(raster_files)}")

if not raster_files:
    print("No raster files found referenced in the VRT. Exiting.")
    sys.exit(1)

# ===========================
# LOAD BOUNDARY SHAPEFILE
# ===========================

boundary_shapefile_path = os.path.join(
    base_dir, "data/multi-domain-data/boundary-layers/processed/admin-boundary-layers/wwri_study_area_admin_0.shp"
)
required_extensions = ['.shx', '.dbf', '.prj']
base_name = os.path.splitext(boundary_shapefile_path)[0]
missing_files = [base_name + ext for ext in required_extensions if not os.path.exists(base_name + ext)]

if not os.path.exists(boundary_shapefile_path):
    print(f"Shapefile does not exist at the path: {boundary_shapefile_path}")
    sys.exit(1)

if missing_files:
    print(f"Missing shapefile components: {missing_files}")
    sys.exit(1)

print("Loading boundary shapefile...")
try:
    study_area = gpd.read_file(boundary_shapefile_path, engine='fiona')
    print("Shapefile loaded successfully.")
except Exception as e:
    print(f"Error reading shapefile: {e}")
    logger.error(f"Error reading shapefile: {e}")
    traceback_str = traceback.format_exc()
    logger.error(traceback_str)
    sys.exit(1)

# ===========================
# REPROJECT STUDY AREA TO MATCH RASTER CRS
# ===========================

print("Reprojecting boundary to match raster CRS...")
try:
    # Find the first valid raster to determine CRS
    valid_raster_found = False
    for raster in raster_files:
        if os.path.exists(raster):
            with rasterio.open(raster) as src:
                raster_crs = src.crs
            valid_raster_found = True
            break
    if not valid_raster_found:
        print("No valid rasters found to determine CRS. Exiting.")
        sys.exit(1)
    
    study_area = study_area.to_crs(raster_crs)
    print(f"Reprojected study area to CRS: {raster_crs}")
except Exception as e:
    print(f"Error determining or reprojecting CRS: {e}")
    logger.error(f"Error determining or reprojecting CRS: {e}")
    traceback_str = traceback.format_exc()
    logger.error(traceback_str)
    sys.exit(1)

# ===========================
# CLASSIFY RASTERS BASED ON THEIR RELATION TO STUDY AREA
# ===========================

print("Classifying rasters (fully inside, edge, outside)...")
fully_inside_rasters = []
edge_rasters = []

# Create a unary union of the study area for efficient spatial operations
# Note: As of GeoPandas 0.10, unary_union is deprecated in favor of union_all()
# Adjust based on your GeoPandas version
try:
    study_union = study_area.geometry.unary_union
except AttributeError:
    study_union = study_area.geometry.union_all()

for raster_path in tqdm(raster_files, desc="Checking bounding boxes"):
    try:
        if not os.path.exists(raster_path):
            logger.error(f"Raster file does not exist: {raster_path}")
            continue
        with rasterio.open(raster_path) as src:
            bounds = src.bounds
            raster_bbox = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
            # Check if bounding box is fully inside the study area
            if raster_bbox.within(study_union):
                fully_inside_rasters.append(raster_path)
            # Check if it intersects (but not fully within)
            elif raster_bbox.intersects(study_union):
                edge_rasters.append(raster_path)
            # Else, it's fully outside; do nothing
    except Exception as e:
        logger.error(f"Error processing raster {raster_path}: {e}")
        traceback_str = traceback.format_exc()
        logger.error(traceback_str)

print(f"Fully inside rasters: {len(fully_inside_rasters)}")
print(f"Edge rasters: {len(edge_rasters)}")

if not (fully_inside_rasters or edge_rasters):
    print("No rasters intersect the study area at all. Exiting.")
    sys.exit(1)

# ===========================
# MASK ONLY THE EDGE RASTERS IN PARALLEL
# ===========================

print("Masking edge rasters so cells outside the boundary are set to NA...")
masked_dir = os.path.join(os.path.dirname(vrt_path), MASKED_DIR_NAME)
os.makedirs(masked_dir, exist_ok=True)
masked_raster_paths = []

if edge_rasters:
    study_geometry = [geom for geom in study_area.geometry]
    
    # Determine nodata value from the first edge raster; default to -9999 if not set
    nodata = -9999
    for raster in edge_rasters:
        try:
            with rasterio.open(raster) as src:
                if src.nodata is not None:
                    nodata = src.nodata
                break
        except Exception as e:
            logger.error(f"Error determining nodata value from raster {raster}: {e}")
            traceback_str = traceback.format_exc()
            logger.error(traceback_str)
            continue

    def mask_raster(raster_path):
        try:
            with rasterio.open(raster_path) as src:
                out_image, out_transform = rasterio.mask.mask(
                    src,
                    study_geometry,
                    crop=False,
                    nodata=nodata,
                    filled=True,
                    invert=False
                )
                out_meta = src.meta.copy()
                out_meta.update({
                    "height": out_image.shape[1],
                    "width": out_image.shape[2],
                    "transform": out_transform,
                    "nodata": nodata,
                    "BIGTIFF": 'YES'  # Enable BIGTIFF
                })
                
                # Generate unique output path
                # Incorporate parent directory names to ensure uniqueness
                relative_path = os.path.relpath(raster_path, os.path.dirname(vrt_path))
                # Replace path separators with underscores to flatten the directory structure
                unique_name = relative_path.replace(os.sep, '_')
                masked_raster_name = f"masked_{unique_name}"
                masked_raster_path = os.path.join(masked_dir, masked_raster_name)
                
                # Ensure the directory exists (if mirroring directory structure)
                os.makedirs(os.path.dirname(masked_raster_path), exist_ok=True)
                
                with rasterio.open(masked_raster_path, "w", **out_meta) as dest:
                    dest.write(out_image)
                
                return masked_raster_path
        except Exception as e:
            logger.error(f"Error masking raster {raster_path}: {e}")
            traceback_str = traceback.format_exc()
            logger.error(traceback_str)
            return None

    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        # Submit all masking tasks
        futures = {executor.submit(mask_raster, raster_path): raster_path for raster_path in edge_rasters}
        # Use tqdm to display progress
        for future in tqdm(as_completed(futures), total=len(futures), desc="Masking edge rasters"):
            raster_path = futures[future]
            try:
                result = future.result()
                if result:
                    masked_raster_paths.append(result)
            except Exception as e:
                logger.error(f"Unhandled exception for raster {raster_path}: {e}")
                traceback_str = traceback.format_exc()
                logger.error(traceback_str)

    print(f"Masked rasters created: {len(masked_raster_paths)}")
else:
    print("No edge rasters to mask.")

# ===========================
# PREPARE LIST OF RASTERS TO BE INCLUDED IN THE NEW VRT
# ===========================

rasters_for_vrt = fully_inside_rasters + masked_raster_paths
print(f"Total rasters to include in the new VRT: {len(rasters_for_vrt)}")

if not rasters_for_vrt:
    print("No rasters to reference in the new VRT. Exiting.")
    sys.exit(1)

# ===========================
# CREATE A NEW VRT FILE REFERENCING ONLY THE SELECTED RASTERS
# ===========================

# Define the new VRT directory and ensure it exists
os.makedirs(NEW_VRT_DIR, exist_ok=True)

# Define the new VRT path with the specified name
new_vrt_path = os.path.join(NEW_VRT_DIR, NEW_VRT_NAME)
print(f"Creating new VRT at: {new_vrt_path}")

try:
    # Write all raster paths to a temporary file to handle large number of rasters
    temp_raster_list = "/tmp/raster_list.txt"
    with open(temp_raster_list, "w") as list_file:
        for raster in rasters_for_vrt:
            list_file.write(f"{raster}\n")
    
    # Construct the command with input file list
    command = ["gdalbuildvrt", "-input_file_list", temp_raster_list, new_vrt_path]
    subprocess.run(command, check=True)
    
    # Remove the temporary raster list file
    os.remove(temp_raster_list)
    
    print(f"New VRT created: {new_vrt_path}")
except subprocess.CalledProcessError as e:
    print(f"Error creating new VRT: {e}")
    logger.error(f"Error creating new VRT: {e}")
    traceback_str = traceback.format_exc()
    logger.error(traceback_str)
    sys.exit(1)
except Exception as e:
    print(f"Unexpected error during VRT creation: {e}")
    logger.error(f"Unexpected error during VRT creation: {e}")
    traceback_str = traceback.format_exc()
    logger.error(traceback_str)
    sys.exit(1)

# ===========================
# PLOT BOUNDING BOXES OF RASTERS
# ===========================

print("Plotting bounding boxes of fully inside and edge rasters...")

# Function to create GeoDataFrame from raster paths
def create_geodataframe(raster_paths, classification):
    geometries = []
    for raster in raster_paths:
        try:
            with rasterio.open(raster) as src:
                bounds = src.bounds
                geom = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
                geometries.append(geom)
        except Exception as e:
            logger.error(f"Error reading raster for plotting {raster}: {e}")
            traceback_str = traceback.format_exc()
            logger.error(traceback_str)
    gdf = gpd.GeoDataFrame({'geometry': geometries, 'classification': classification}, crs=raster_crs)
    return gdf

# Create GeoDataFrames
fully_inside_gdf = create_geodataframe(fully_inside_rasters, 'Fully Inside')
edge_gdf = create_geodataframe(edge_rasters, 'Edge')

# Combine GeoDataFrames
combined_gdf = gpd.GeoDataFrame(pd.concat([fully_inside_gdf, edge_gdf], ignore_index=True), crs=raster_crs)

# Plotting
try:
    fig, ax = plt.subplots(figsize=(12, 12))
    
    # Plot study area boundary
    study_area.boundary.plot(ax=ax, edgecolor='black', linewidth=1, label='Study Area Boundary')
    
    # Plot fully inside rasters
    fully_inside_gdf.plot(ax=ax, facecolor='none', edgecolor='green', linewidth=0.5, label='Fully Inside Rasters')
    
    # Plot edge rasters
    edge_gdf.plot(ax=ax, facecolor='none', edgecolor='red', linewidth=0.5, label='Edge Rasters')
    
    # Add legend
    plt.legend()
    
    # Add title
    plt.title('Bounding Boxes of Fully Inside and Edge Rasters')
    
    # Save the plot
    plot_output_path = os.path.join(PLOT_OUTPUT_DIR, PLOT_OUTPUT_NAME)
    plt.savefig(plot_output_path, dpi=300)
    plt.close()
    
    print(f"Bounding boxes plot saved at: {plot_output_path}")
except Exception as e:
    print(f"Error during plotting: {e}")
    logger.error(f"Error during plotting: {e}")
    traceback_str = traceback.format_exc()
    logger.error(traceback_str)

# ===========================
# PROCESS COMPLETION MESSAGE
# ===========================

print("Process completed successfully.")
print(f"Any errors encountered have been logged to {LOG_FILE}.")
